In [1]:
import numpy as np
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from collections import defaultdict
import warnings
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist

warnings.filterwarnings('ignore')

In [2]:
gmt = {}
with open('/Users/anna/Projects/Gene-Knowledge-Graph/public/aging_atlas_gtex.gmt', 'r') as f:
    for line in f:
        tks = line.split("\t")
        gmt[tks[0]] = tks[2:]

In [3]:
def get_cheakg_results(chea_gene_list, desc="", term_limit=10):
    '''
    Find the subnetwork of enriched TFs for an input gene list
    '''
    CHEA_KG = 'https://chea-kg.maayanlab.cloud/api/enrichment'
        
    payload = {
        'list': (None, "\n".join(chea_gene_list)),
        'description': (None, desc)
    }
    try:
        response=requests.post(f"{CHEA_KG}/addList", files=payload)
        data = json.loads(response.text)
    except Exception as e: 
        print("Error connecting to ChEA-KG: ", e)
    
    q = {
        'min_lib': 3, # minimum number of libraries that a TF must be ranked in
        'libraries': [
            {'library': "Integrated--meanRank", 'term_limit': term_limit} # edit term_limit to change number of top-ranked TFs
        ],
        'limit':50, # controls number of edges returned - may cause issues with visualization if too large
        'userListId': data['userListId']
    }
    
    query_json=json.dumps(q)
    
    res = requests.post(CHEA_KG, data=query_json)
    if res.ok:
        data = json.loads(res.text)
    else:
        data = None
        print(res.text)
    return data

In [4]:
networks = {}
for term, gs in gmt.items():
    networks[term] = get_cheakg_results(gs, desc=term, term_limit=10)

In [5]:
with open("./aging-atlas-subnetworks.json", 'w') as f:
    json.dump(networks, f)

In [20]:
networks

{'Adipose Tissue:down': {'nodes': [{'data': {'id': 266743,
     'kind': 'Top Ranked TFs',
     'label': 'NPAS4',
     'HGNC': 'HGNC:18983',
     'Ensembl': 'ENSG00000174576',
     'uri': 'https://www.ncbi.nlm.nih.gov/gene/266743',
     'library': 'Integrated--meanRank',
     'enrichr_label': 'NPAS4',
     'score': 70.33,
     'rank': '14',
     'overlap': 18,
     'libs': [{'library': 'ARCHS4 Coexpression', 'score': 67},
      {'library': 'Enrichr Queries', 'score': 136},
      {'library': 'GTEx Coexpression', 'score': 8}],
     'rank_sum': 211,
     'value': 0.3104901960784314,
     'node_type': 0,
     'borderWidth': 0,
     'gradient_color': '#8ad6ff',
     'color': '#8ad6ff'}},
   {'data': {'id': 30813,
     'kind': 'Queried TFs that are also enriched',
     'label': 'VSX1',
     'HGNC': 'HGNC:12723',
     'Ensembl': 'ENSG00000100987',
     'uri': 'https://www.ncbi.nlm.nih.gov/gene/30813',
     'library': 'Integrated--meanRank',
     'enrichr_label': 'VSX1',
     'score': 36,
     

In [44]:
node_agg = {}
up_node_agg = {}
dn_node_agg = {}

edge_agg = {}
up_edge_agg = {}
dn_edge_agg = {}

for network_name, network_data in networks.items():
    
    # ---- NODES ----
    for node in network_data["nodes"]:
        d = node["data"]
        node_id = str(d["id"])
        
        if node_id not in node_agg:
            node_agg[node_id] = {
                "id": node_id,
                "label": d["label"],
                "count": 0,
                "networks": set()
            }
        
        node_agg[node_id]["count"] += 1
        node_agg[node_id]["networks"].add(network_name)

        if "up" in network_name:
            if node_id not in up_node_agg:
                up_node_agg[node_id] = {
                    "id": node_id,
                    "label": d["label"],
                    "count": 0,
                    "networks": set()
                }
        
            up_node_agg[node_id]["count"] += 1
            up_node_agg[node_id]["networks"].add(network_name)

        if "down" in network_name:
            if node_id not in dn_node_agg:
                dn_node_agg[node_id] = {
                    "id": node_id,
                    "label": d["label"],
                    "count": 0,
                    "networks": set()
                }
        
            dn_node_agg[node_id]["count"] += 1
            dn_node_agg[node_id]["networks"].add(network_name)       
    
    
    # ---- EDGES ----
    for edge in network_data["edges"]:
        d = edge["data"]
        
        source = str(d["source"])
        target = str(d["target"])
        relation = d["relation"]
        
        edge_key = (source, target, relation)
        
        if edge_key not in edge_agg:
            edge_agg[edge_key] = {
                "id": f"{source}_{target}_{relation}",
                "source": source,
                "target": target,
                "source_label": d["source_label"],
                "target_label": d["target_label"],
                "relation": relation,
                "count": 0,
                "networks": set()
            }
        
        edge_agg[edge_key]["count"] += 1
        edge_agg[edge_key]["networks"].add(network_name)

        if "up" in network_name:
            if edge_key not in up_edge_agg:
                up_edge_agg[edge_key] = {
                    "id": f"{source}_{target}_{relation}",
                    "source": source,
                    "target": target,
                    "source_label": d["source_label"],
                    "target_label": d["target_label"],
                    "relation": relation,
                    "count": 0,
                    "networks": set()
                }
            
            up_edge_agg[edge_key]["count"] += 1
            up_edge_agg[edge_key]["networks"].add(network_name)

        if "down" in network_name:
            if edge_key not in dn_edge_agg:
                dn_edge_agg[edge_key] = {
                    "id": f"{source}_{target}_{relation}",
                    "source": source,
                    "target": target,
                    "source_label": d["source_label"],
                    "target_label": d["target_label"],
                    "relation": relation,
                    "count": 0,
                    "networks": set()
                }
            
            dn_edge_agg[edge_key]["count"] += 1
            dn_edge_agg[edge_key]["networks"].add(network_name)

In [45]:
def scale_node_radius(count, min_r=30):
    return min_r + (count - 1) * 8

def scale_edge_width(count, min_w=0.5):
    return min_w + (count - 1) * 0.5

In [46]:
cy_nodes = []
cy_edges = []

# Nodes
for node in node_agg.values():
    cy_nodes.append({
        "data": {
            "id": node["id"],
            "label": node["label"],
            "radius": scale_node_radius(node["count"]),
            "count": node["count"],
            "networks": sorted(list(node["networks"]))
        }
    })

# Edges
for edge in edge_agg.values():
    cy_edges.append({
        "data": {
            "id": edge["id"],
            "source": edge["source"],
            "target": edge["target"],
            "source_label": edge["source_label"],
            "target_label": edge["target_label"],
            "relation": edge["relation"],
            "width": scale_edge_width(edge["count"]),
            "count": edge["count"],
            "networks": sorted(list(edge["networks"]))
        }
    })

cytoscape_json = {
    "elements": {
        "nodes": cy_nodes,
        "edges": cy_edges
    }
}

In [47]:
up_cy_nodes = []
up_cy_edges = []

# Nodes
for node in up_node_agg.values():
    up_cy_nodes.append({
        "data": {
            "id": node["id"],
            "label": node["label"],
            "radius": scale_node_radius(node["count"]),
            "count": node["count"],
            "networks": sorted(list(node["networks"]))
        }
    })

# Edges
for edge in up_edge_agg.values():
    up_cy_edges.append({
        "data": {
            "id": edge["id"],
            "source": edge["source"],
            "target": edge["target"],
            "source_label": edge["source_label"],
            "target_label": edge["target_label"],
            "relation": edge["relation"],
            "width": scale_edge_width(edge["count"]),
            "count": edge["count"],
            "networks": sorted(list(edge["networks"]))
        }
    })

up_cytoscape_json = {
    "elements": {
        "nodes": up_cy_nodes,
        "edges": up_cy_edges
    }
}

In [48]:
down_cy_nodes = []
down_cy_edges = []

# Nodes
for node in dn_node_agg.values():
    down_cy_nodes.append({
        "data": {
            "id": node["id"],
            "label": node["label"],
            "radius": scale_node_radius(node["count"]),
            "count": node["count"],
            "networks": sorted(list(node["networks"]))
        }
    })

# Edges
for edge in dn_edge_agg.values():
    down_cy_edges.append({
        "data": {
            "id": edge["id"],
            "source": edge["source"],
            "target": edge["target"],
            "source_label": edge["source_label"],
            "target_label": edge["target_label"],
            "relation": edge["relation"],
            "width": scale_edge_width(edge["count"]),
            "count": edge["count"],
            "networks": sorted(list(edge["networks"]))
        }
    })

down_cytoscape_json = {
    "elements": {
        "nodes": down_cy_nodes,
        "edges": down_cy_edges
    }
}

In [49]:
with open("aging_atlas_test_json.json", 'w') as f:
    json.dump(cytoscape_json, f)

In [50]:
with open("aging_atlas_test_up.json", 'w') as f:
    json.dump(up_cytoscape_json, f)

In [51]:
with open("aging_atlas_test_dn.json", 'w') as f:
    json.dump(down_cytoscape_json, f)

In [52]:
up_cytoscape_json

{'elements': {'nodes': [{'data': {'id': '3664',
     'label': 'IRF6',
     'radius': 38,
     'count': 2,
     'networks': ['Adipose Tissue:up', 'Ovary:up']}},
   {'data': {'id': '57822',
     'label': 'GRHL3',
     'radius': 46,
     'count': 3,
     'networks': ['Adipose Tissue:up', 'Breast:up', 'Ovary:up']}},
   {'data': {'id': '79977',
     'label': 'GRHL2',
     'radius': 38,
     'count': 2,
     'networks': ['Adipose Tissue:up', 'Blood Vessel:up']}},
   {'data': {'id': '8456',
     'label': 'FOXN1',
     'radius': 70,
     'count': 6,
     'networks': ['Adipose Tissue:up',
      'Breast:up',
      'Kidney:up',
      'Ovary:up',
      'Pancreas:up',
      'Vagina:up']}},
   {'data': {'id': '29841',
     'label': 'GRHL1',
     'radius': 30,
     'count': 1,
     'networks': ['Adipose Tissue:up']}},
   {'data': {'id': '50805',
     'label': 'IRX4',
     'radius': 38,
     'count': 2,
     'networks': ['Adipose Tissue:up', 'Kidney:up']}},
   {'data': {'id': '8626',
     'label': 'TP

In [53]:
up_nodes = pd.DataFrame([node['data'] for node in up_cytoscape_json['elements']['nodes']])
up_nodes.sort_values(by='count', ascending=False)

,id,label,radius,count,networks
34,91464,ISX,70,6,"[Blood Vessel:up, Breast:up, Liver:up, Lung:up..."
3,8456,FOXN1,70,6,"[Adipose Tissue:up, Breast:up, Kidney:up, Ovar..."
18,79190,IRX6,62,5,"[Blood:up, Lung:up, Pancreas:up, Pituitary:up,..."
31,347853,TBX10,54,4,"[Blood Vessel:up, Breast:up, Skin:up, Thyroid:up]"
97,116448,OLIG1,54,4,"[Nerve:up, Small Intestine:up, Spleen:up, Vagi..."
...,...,...,...,...,...
77,6097,RORC,30,1,[Lung:up]
21,8856,NR1I2,30,1,[Blood:up]
22,121599,SPIC,30,1,[Blood:up]
73,29842,TFCP2L1,30,1,[Liver:up]


In [54]:
up_nodes['count'].value_counts()

count
1    82
2    31
3    18
4     5
6     2
5     1
Name: count, dtype: int64

In [55]:
up_edges = pd.DataFrame([node['data'] for node in up_cytoscape_json['elements']['edges']])
up_edges.sort_values(by='count', ascending=False)

,id,source,target,source_label,target_label,relation,width,count,networks
2,8456_8456_downregulates,8456,8456,FOXN1,FOXN1,downregulates,3.0,6,"[Adipose Tissue:up, Breast:up, Kidney:up, Ovar..."
75,91464_91464_downregulates,91464,91464,ISX,ISX,downregulates,3.0,6,"[Blood Vessel:up, Breast:up, Liver:up, Lung:up..."
34,5468_5468_upregulates,5468,5468,PPARG,PPARG,upregulates,2.0,4,"[Blood:up, Lung:up, Pituitary:up, Small Intest..."
4,79755_79755_upregulates,79755,79755,ZNF750,ZNF750,upregulates,2.0,4,"[Adipose Tissue:up, Ovary:up, Pancreas:up, Sto..."
252,116448_116448_upregulates,116448,116448,OLIG1,OLIG1,upregulates,2.0,4,"[Nerve:up, Small Intestine:up, Spleen:up, Vagi..."
...,...,...,...,...,...,...,...,...,...
173,8456_25833_upregulates,8456,25833,FOXN1,POU2F3,upregulates,0.5,1,[Kidney:up]
172,50805_5013_upregulates,50805,5013,IRX4,OTX1,upregulates,0.5,1,[Kidney:up]
171,1045_221833_upregulates,1045,221833,CDX2,SP8,upregulates,0.5,1,[Kidney:up]
170,8456_50805_downregulates,8456,50805,FOXN1,IRX4,downregulates,0.5,1,[Kidney:up]
